In [2]:
#sqlite -> Motor de base de datos.

import sqlite3
import pandas as pd
from pathlib import Path #Manejo de rutas
print("Entorno Listo")

Entorno Listo


In [3]:
#Conexion con sqlite y cursor
#cursor -> El que ejecuta los comandos sql
#conn -> la puerta abierta con la base de datos

conn = sqlite3.connect(':memory:') #RAM
cursor = conn.cursor()

print("Conexión con sqlite en memoria establecida")


Conexión con sqlite en memoria establecida


In [8]:
#cargar los csv como tablas SQLite
df_candidatos = pd.read_csv("candidatos.csv")
df_ofertas = pd.read_csv("ofertas.csv")
df_postulaciones = pd.read_csv("postulaciones.csv")

df_candidatos.to_sql("candidatos", conn, if_exists='replace', index=False)
df_ofertas.to_sql("ofertas", conn, if_exists='replace', index=False)
df_postulaciones.to_sql("postulaciones", conn, if_exists='replace', index=False)

print(f"candidatos  -> {len(df_candidatos):,} filas | columnas: {df_candidatos.columns.tolist()}")

candidatos  -> 1,200 filas | columnas: ['candidato_id', 'nombre', 'ciudad', 'area_interes', 'nivel', 'anos_experiencia', 'skills_principales', 'salario_esperado_cop', 'disponible', 'fecha_registro']


In [9]:
#Función auxiliar
# display -> muestra el Dataframe como formato de tabla HTML

def ejecutar_sql(query, limite_display=20):
    #ejecuta una query SQL y muestra el resultado como tabla.
    resultado = pd.read_sql_query(query, conn)
    print(f"{len(resultado)} filas x {len(resultado.columns)} columnas")
    return resultado.head(limite_display)

print("Función ejecutar_sql() lista")

Función ejecutar_sql() lista


In [12]:
#SELECT: Qué columnas quiero ver
#FROM: De Donde
#Limit Maximos cuantas filas
#Limit 5 evitas que traiga los 1.200 filas

query_01 = """
SELECT *
FROM candidatos
LIMIT 5
"""
ejecutar_sql(query_01)

5 filas x 10 columnas


,candidato_id,nombre,ciudad,area_interes,nivel,anos_experiencia,skills_principales,salario_esperado_cop,disponible,fecha_registro
0,1,Natalia López,Montería,Data Science,Senior,6,TensorFlow|Scikit-learn|Python|SQL,7785972,Sí,2024-09-15
1,2,Alejandro García,Montería,Cloud,Semi-Senior,5,GCP|AWS|Azure|Kubernetes,5165476,Sí,2024-02-22
2,3,Camila Moreno,Pasto,Backend,Senior,5,Python|SQL|Spring Boot|Node.js,7583470,Sí,2024-05-28
3,4,Camila Molina,Pereira,DevOps,Junior,1,Jenkins|Linux,3405459,Sí,2024-09-30
4,5,Miguel Ramírez,Ibagué,Frontend,Lead,11,Vue.js|JavaScript|React|CSS,11769852,Sí,2024-04-18


In [17]:
#ORDER BY: Agrupar por

query_02 = """
SELECT
    nombre,
    ciudad,
    area_interes,
    nivel
FROM candidatos
ORDER BY ciudad ASC
LIMIT 10
"""
ejecutar_sql(query_02)


10 filas x 4 columnas


,nombre,ciudad,area_interes,nivel
0,Luisa Alvarado,Armenia,Machine Learning,Lead
1,David Romero,Armenia,Machine Learning,Senior
2,Valentina Martínez,Armenia,BI & Analytics,Lead
3,Gabriela Flores,Armenia,Ciberseguridad,Junior
4,Paula Ortiz,Armenia,Machine Learning,Lead
5,Nicolás González,Armenia,Full Stack,Semi-Senior
6,Mateo González,Armenia,Full Stack,Senior
7,Camila Ramos,Armenia,Data Science,Senior
8,Santiago Guerrero,Armenia,Backend,Junior
9,David Guerrero,Armenia,Full Stack,Lead


In [18]:
#DISTINCT: Valores unicos

# Ciudades unicas de candidatos:
query_03a = """
SELECT DISTINCT ciudad
FROM candidatos
ORDER BY ciudad ASC
"""
ejecutar_sql(query_03a)


18 filas x 1 columnas


,ciudad
0,Armenia
1,Barranquilla
2,Bogotá
3,Bucaramanga
4,Cali
5,Cartagena
6,Cúcuta
7,Ibagué
8,Manizales
9,Medellín


In [19]:
#DISTINCT: areas de interés

query_03b = """
SELECT DISTINCT area_interes
FROM candidatos
ORDER BY area_interes ASC
"""

ejecutar_sql(query_03b)


12 filas x 1 columnas


,area_interes
0,BI & Analytics
1,Backend
2,Ciberseguridad
3,Cloud
4,Data Science
5,DevOps
6,Diseño UX/UI
7,Frontend
8,Full Stack
9,Machine Learning


In [22]:
#ALIAS
#AS le da un nombre más legible a las columnas en el resultado

query_04 = """
SELECT
    nombre                 AS "Candidato",
    ciudad                 AS "Ciudad",
    area_interes           AS "Área",
    salario_esperado_cop   AS "Salario Esperado (COP)"
FROM candidatos
ORDER BY salario_esperado_cop DESC
LIMIT 8
"""

ejecutar_sql(query_04)




8 filas x 4 columnas


,Candidato,Ciudad,Área,Salario Esperado (COP)
0,Juliana Jiménez,Cali,Frontend,17999255
1,Ana Jiménez,Villavicencio,Frontend,17992576
2,Ana Castro,Villavicencio,DevOps,17960879
3,Diego Suárez,Medellín,Full Stack,17959308
4,Santiago Mendoza,Cartagena,BI & Analytics,17921070
5,Camila Flores,Neiva,Machine Learning,17842340
6,Nicolás Pérez,Villavicencio,Machine Learning,17778316
7,Santiago Martínez,Tunja,Mobile,17764740


JOIN: CURZAR LOS DATOS

¿QUIÉN SE POSTULÓ A QUÉ OFERTA?

In [24]:
#INNER JOIN - LA UNIÓN MÁS COMÚN
#RETORNA SOLO LAS FILAS QUE TIENEN COINCIDENCIA EN AMBAS TABLAS
#POSTULACIONES

query_10 = """
SELECT 
    p.postulacion_id,
    c.nombre           AS candidatos,
    c.ciudad,
    c.area_interes,
    p.estado_postulacion,
    p.fecha_postulacion
FROM postulaciones p
INNER JOIN candidatos c ON p.candidato_id = c.candidato_id
ORDER BY p.fecha_postulacion DESC
LIMIT 20
"""
ejecutar_sql(query_10)


20 filas x 6 columnas


,postulacion_id,candidatos,ciudad,area_interes,estado_postulacion,fecha_postulacion
0,106,María González,Neiva,Cloud,En revisión,2024-11-27
1,420,Laura García,Medellín,Mobile,Rechazada,2024-11-27
2,797,Paula Rodríguez,Cúcuta,Machine Learning,En revisión,2024-11-27
3,1072,Camila López,Popayán,Cloud,Rechazada,2024-11-27
4,1306,Ana Romero,Manizales,Backend,Enviada,2024-11-27
5,1496,Daniel Ramos,Pasto,Full Stack,En revisión,2024-11-27
6,153,Juliana Suárez,Popayán,Mobile,Enviada,2024-11-26
7,162,Paula Suárez,Tunja,Data Science,En revisión,2024-11-26
8,263,David Castro,Cartagena,Machine Learning,Entrevista,2024-11-26
9,568,Carolina Romero,Ibagué,Backend,Enviada,2024-11-26
